# Construire un projet de Machine Learning:

## Partie 3 | `Algorithmes de Machine Learning`

Dans ce notebook, nous allons préparer le jeu de données des ours pour la construction de modèles de machine learning.

### Ce que nous allons couvrir :

1. **Chargement des données** - Charger le jeu de données des ours avec Modin (`modin.pandas`) et Snowpark (`snowflake-snowpark-python`)
2. **Préparation des données** - Mettre les variables à l’échelle et préparer les données pour l’entraînement des modèles avec `scikit-learn`
3. **Entraînement des modèles** - Entraîner plusieurs modèles de machine learning avec `scikit-learn` :
   - Régression logistique (`LogisticRegression`)
   - Forêt aléatoire (`RandomForestClassifier`)
   - Machine à vecteurs de support (`SVC`)
4. **Comparaison des performances** - Comparer les modèles avec la précision et la métrique MCC (`scikit-learn`)
5. **Interprétabilité des modèles** - Analyser l’importance des variables et les coefficients des modèles pour comprendre les prédictions (`Altair`)


## Installer les bibliothèques prérequises

Les Snowflake Notebooks incluent par défaut les bibliothèques Python courantes. Pour en ajouter d’autres, utilisez le menu déroulant **Packages** en haut à droite. 

Ajoutons les packages suivants :
- `modin` - Effectuer des opérations sur les données (lecture/écriture) et du data wrangling comme avec pandas via l’[API Snowpark pandas](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/modin/index)
- `scikit-learn` - Effectuer des divisions de données et construire des modèles de machine learning
- `snowflake-ml-python` - un ensemble de fonctionnalités ML fournies par Snowflake. Ici, nous utiliserons la fonctionnalité de journalisation des métriques de modèle.


## 1. Établir la connexion Snowflake

Nous allons commencer par obtenir une session active via la méthode `get_active_session()`.

In [ ]:
# Get active Snowflake session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print(f"✅ Connected using active Snowflake session!")

## 2. Opérations sur les données

Dans cette section, nous allons charger les données, préparer les variables/caractéristiques, explorer les données manquantes et effectuer le découpage des données.

### 2.1. Charger les données

Les données sont lues depuis la table `BEAR` stockée dans Snowflake via la méthode `read_snowflake()`.

In [ ]:
import modin.pandas as pd
import snowflake.snowpark.modin.plugin

bear_df = pd.read_snowflake("BEARS_DATA.STAGES.BEAR")
bear_df

### 2.2. Préparer les variables explicatives et la classe

Le DataFrame est séparé en 2 parties. Les caractéristiques sont assignées à la variable `X` tandis que la classe est assignée à `y`.

In [ ]:
X = bear_df.drop(columns=['species', 'id'])
y = bear_df['species']

### 2.3. Vérifier les données manquantes

In [ ]:
# Data quality checks
missing_features = X.isnull().sum().sum()
missing_target = y.isnull().sum()

print(f"\n🔍 Data Quality:")
print(f"   Missing feature values: {missing_features}")
print(f"   Missing target values: {missing_target}")

### 2.4. Découpage des données
Les données sont séparées en ensembles d’entraînement et de test selon un ratio 80/20 avec `scikit-learn` :
- 80 % sont utilisés comme **ensemble d’entraînement** - utilisé pour entraîner un modèle de ML
- 20 % sont utilisés comme **ensemble de test** - utilisé pour tester le modèle de ML

In [ ]:
# Import scikit-learn modules at first use
from sklearn.model_selection import train_test_split

# Split data using scikit-learn (recommended by Snowflake for ML operations)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Maintain target distribution
)

print("✅ Data splitting completed!")
print('-' * 35) 

print("📊 Data Split Summary:")
print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Number of features: {X_train.shape[1]}")
print('-' * 35) 

# Check class distribution in splits
print("\n🎯 Class Distribution:")
print("Training set:", y_train.value_counts().sort_index().to_dict())
print("Testing set:", y_test.value_counts().sort_index().to_dict())
print('-' * 35) 

### 2.5. Mise à l’échelle des variables

La mise à l’échelle des variables est une technique de prétraitement des données utilisée pour standardiser l’étendue des variables indépendantes ou caractéristiques. Cela permet d’éviter que les variables ayant de plus grandes plages de valeurs (par exemple une variable variant de 10 000 à 1 000 000 tandis que d’autres vont de 0,1 à 0,8) n’influencent de manière disproportionnée le processus d’apprentissage du modèle.

Ici, nous utilisons `scikit-learn` pour mettre les variables à l’échelle en standardisant toutes les variables par centrage sur la moyenne (moyenne = 0) et variance unitaire (écart-type = 1).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Identify numerical and categorical columns
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

print("Numerical features:", numerical_features.tolist())
print("Categorical features:", categorical_features.tolist())

# Scale numerical features
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train[numerical_features])
X_test_scaled_num = scaler.transform(X_test[numerical_features])

# Convert categorical features using one-hot encoding
onehot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_scaled_cat = onehot.fit_transform(X_train[categorical_features])
X_test_scaled_cat = onehot.transform(X_test[categorical_features])

# Get feature names after one-hot encoding
cat_feature_names = onehot.get_feature_names_out(categorical_features)

# Combine numerical and categorical features
X_train_scaled = np.hstack([X_train_scaled_num, X_train_scaled_cat])
X_test_scaled = np.hstack([X_test_scaled_num, X_test_scaled_cat])

# Convert to DataFrame with proper column names
all_feature_names = list(numerical_features) + list(cat_feature_names)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=all_feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=all_feature_names, index=X_test.index)

print("\n✅ Features scaling completed!")
print('-' * 35) 

print("\n📊 Scaled Data Dimension:")
print(f"Scaled training features shape: {X_train_scaled.shape}")
print(f"Scaled testing features shape: {X_test_scaled.shape}")
print('-' * 35) 

# Show scaling effect for numerical features
if len(numerical_features) > 0:
    first_num_feature = numerical_features[0]
    print("\n📊 Scaling Effect (first numerical feature):")
    print(f"Original {first_num_feature}: mean={X_train[first_num_feature].mean():.3f}, std={X_train[first_num_feature].std():.3f}")
    print(f"Scaled {first_num_feature}: mean={X_train_scaled[first_num_feature].mean():.3f}, std={X_train_scaled[first_num_feature].std():.3f}")
print('-' * 35)


## 3. Entraînement des modèles de machine learning
Maintenant que nous avons des variables mises à l’échelle, nous allons construire des modèles de ML avec `scikit-learn`.

### 3.1. Régression logistique


In [ ]:
# Import logistic model and classification metrics
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, matthews_corrcoef
import numpy as np

# Logistic Regression using scikit-learn
print("🔧 Training Logistic Regression Model...")

log_reg_model = LogisticRegression(random_state=42) # random_state for reproducibility
log_reg_model.fit(X_train_scaled, y_train)

# Make predictions (outputs class labels directly)
log_reg_train_pred = log_reg_model.predict(X_train_scaled)
log_reg_test_pred = log_reg_model.predict(X_test_scaled)

# Calculate classification metrics
logreg_train_acc = accuracy_score(y_train, log_reg_train_pred)
logreg_test_acc = accuracy_score(y_test, log_reg_test_pred)
logreg_train_mcc = matthews_corrcoef(y_train, log_reg_train_pred)
logreg_test_mcc = matthews_corrcoef(y_test, log_reg_test_pred)

test_class_report = classification_report(y_test, log_reg_test_pred)

print("✅ Logistic Regression model trained!")
print('-' * 35)

print(f"📊 Logistic Regression Results:")
print(f"   Training Accuracy: {logreg_train_acc:.4f}")
print(f"   Testing Accuracy:  {logreg_test_acc:.4f}")
print(f"   Training MCC:      {logreg_train_mcc:.4f}")
print(f"   Testing MCC:       {logreg_test_mcc:.4f}")
print('-' * 35)

print("\nClassification Report (Test Set):")
print(test_class_report)
print('-' * 35)

### 3.2. Classifieur Forêt aléatoire


In [ ]:
# Import ensemble methods and detailed metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, matthews_corrcoef, classification_report
import pandas as pd

# Random Forest using scikit-learn
print("🌲 Training Random Forest Classifier...")
print('-' * 35)

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1) # Use parallel processing
rf_model.fit(X_train_scaled, y_train)

# Make predictions
rf_train_pred = rf_model.predict(X_train_scaled)
rf_test_pred = rf_model.predict(X_test_scaled)

# Calculate comprehensive metrics
rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)
rf_train_mcc = matthews_corrcoef(y_train, rf_train_pred)
rf_test_mcc = matthews_corrcoef(y_test, rf_test_pred)
test_class_report = classification_report(y_test, rf_test_pred)

print("✅ Random Forest model trained!")
print('-' * 35)

print(f"📊 Random Forest Results:")
print(f"   Training Accuracy: {rf_train_acc:.4f}")
print(f"   Testing Accuracy:  {rf_test_acc:.4f}")
print(f"   Training MCC:      {rf_train_mcc:.4f}")
print(f"   Testing MCC:       {rf_test_mcc:.4f}")
print("\nClassification Report (Test Set):")
print(test_class_report)
print('-' * 35)

### 3.3. Machine à vecteurs de support (SVM)


In [ ]:
# Import SVM and detailed metrics
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, matthews_corrcoef, classification_report

# Support Vector Machine using scikit-learn
print("🤖 Training Support Vector Machine...")
print('-' * 35)

svm_model = SVC(random_state=42)
svm_model.fit(X_train_scaled, y_train)

# Make predictions
svm_train_pred = svm_model.predict(X_train_scaled)
svm_test_pred = svm_model.predict(X_test_scaled)

# Calculate comprehensive metrics
svm_train_acc = accuracy_score(y_train, svm_train_pred)
svm_test_acc = accuracy_score(y_test, svm_test_pred)
svm_train_mcc = matthews_corrcoef(y_train, svm_train_pred)
svm_test_mcc = matthews_corrcoef(y_test, svm_test_pred)
test_class_report = classification_report(y_test, svm_test_pred)

print("✅ SVM model trained!")
print('-' * 35)

print(f"📊 SVM Results:")
print(f"   Training Accuracy: {svm_train_acc:.4f}")
print(f"   Testing Accuracy:  {svm_test_acc:.4f}")
print(f"   Training MCC:      {svm_train_mcc:.4f}")
print(f"   Testing MCC:       {svm_test_mcc:.4f}")
print("\nClassification Report (Test Set):")
print(test_class_report)
print('-' * 35)


## 4. Évaluation comparative des algorithmes de machine learning
L’évaluation comparative signifie essentiellement que nous comparons différents algorithmes de ML pour voir lequel obtient les meilleures performances et/ou lequel est le plus adapté à notre cas d’usage.

Pour sélectionner le meilleur algorithme de ML à utiliser, nous voulons un algorithme qui généralise bien sur de nouvelles données jamais vues et qui peut fournir des informations exploitables.
1. Surapprentissage du modèle : le premier point sur la bonne généralisation à de nouvelles données peut être évalué par le degré avec lequel l’algorithme surapprend les données
2. Interprétabilité du modèle : le second point sur les informations exploitables peut être obtenu en analysant les variables importantes qui contribuent aux prédictions du modèle


### 4.1. Évaluer le surapprentissage

Le surapprentissage mesure à quel point un modèle est meilleur sur les données sur lesquelles il a été entraîné que sur de nouvelles données jamais vues, ce qui indique qu’il a mémorisé du bruit au lieu d’apprendre un motif général.

> $$ Over fitting = Training Performance - Testing Performance $$

Cette formule calcule la baisse de performance lorsque votre modèle passe des données d’entraînement familières à de nouvelles données de test jamais vues.

- Une grande différence signifie que le modèle est surappris : il a simplement mémorisé les exemples d’entraînement au lieu d’apprendre les vrais motifs, et il échoue donc sur de nouvelles données. 👎
- Une faible différence est une bonne chose : cela signifie que le modèle généralise bien. 👍

In [ ]:
# Import Altair at first use
import altair as alt

# Configure Altair for interactive visualizations
alt.data_transformers.enable('json')
alt.theme.enable('opaque')

# Compare all models
model_acc = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'SVM'],
    'Training_Accuracy': [logreg_train_acc, rf_train_acc, svm_train_acc],
    'Testing_Accuracy': [logreg_test_acc, rf_test_acc, svm_test_acc]
})

model_acc['Overfitting'] = model_acc['Training_Accuracy'] - model_acc['Testing_Accuracy']

model_mcc = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'SVM'],
    'Training_MCC': [logreg_train_mcc, rf_train_mcc, svm_train_mcc],
    'Testing_MCC': [logreg_test_mcc, rf_test_mcc, svm_test_mcc]
})

model_mcc['Overfitting'] = model_mcc['Training_MCC'] - model_mcc['Testing_MCC']


print("📊 Model Comparison:")
print('-' * 35)

print("Accuracy:")
print(model_acc.round(4))
print('-' * 35)

print("MCC:")
print(model_mcc.round(4))


### 4.2. Interprétabilité du modèle

Les modèles de ML interprétables sont ceux qui fournissent des coefficients de variables indiquant directement le degré relatif avec lequel elles influencent les valeurs cibles `y`.

Dans les modèles linéaires, cela peut être résumé par l’équation suivante :

> $$y = m_1x_1 + m_2x_2 + ... + b$$

où $$y$$ est la variable cible ou dépendante, $$m_n$$ sont les coefficients des variables, $$x_n$$ sont les variables explicatives ou indépendantes et $$b$$ est la valeur de base.

En essence, les coefficients $$m_n$$ mesurent directement leur influence sur la prédiction de $$y$$, où une valeur absolue plus grande du coefficient signifie un impact plus fort sur la prédiction de $$y$$.



#### 4.2.1. Interpréter les modèles de régression logistique

In [ ]:
# Generated by Snowflake Copilot
# Get coefficients from the model
coefficients = log_reg_model.coef_[0]

# Create a DataFrame using the transformed feature names
logreg_feature_importance = pd.DataFrame({
    'feature': all_feature_names,  # Using all_feature_names from the py_feature_scaling cell
    'coefficient': coefficients
})

# Calculate the absolute value of the coefficients to use as 'importance'
logreg_feature_importance['abs_coefficient'] = np.abs(logreg_feature_importance['coefficient'])

# Sort the features by importance in descending order
logreg_feature_importance = logreg_feature_importance.sort_values('abs_coefficient', ascending=False)

# Print the results
print("✨ Top 5 Most Important Features (Logistic Regression):")
print(logreg_feature_importance[['feature', 'coefficient', 'abs_coefficient']].head())


In [ ]:
# Select the top 5 features for visualization
import altair as alt

top_n = 5
chart_data = logreg_feature_importance.head(top_n)

# Create the bar chart
chart = alt.Chart(chart_data).mark_bar().encode(
    x=alt.X('coefficient:Q', title='Importance'),
    y=alt.Y('feature:N', title='Feature', sort='-x'), # Sort bars by importance
    color=alt.condition(
        alt.datum.coefficient > 0,
        alt.value('#00B8E7'),  # Positive coefficients
        alt.value('#FF61CC')    # Negative coefficients
    ),
    tooltip=[
        alt.Tooltip('feature:N', title='Feature'),
        alt.Tooltip('coefficient:Q', title='Coefficient', format='.4f'),
        alt.Tooltip('importance:Q', title='Importance', format='.4f')
    ]
)

# Explicitly set the text color for axes and titles to black
chart = chart.configure(
    background='transparent'
).configure_axis(
    labelColor='white',  # Color for the feature names and importance values
    titleColor='white'   # Color for the 'Feature' and 'Importance...' titles
).configure_title(
    color='white'        # Color for the main chart title
).properties(
    title=f'Top {top_n} Feature Importance (Logistic Regression)',
    width=600,
    height=400
)

chart

#### 4.2.2. Interpréter les modèles de forêt aléatoire

In [ ]:
# Feature importance analysis with Random forest
rf_feature_importance = pd.DataFrame({
    'feature': all_feature_names, # Using all_feature_names from the py_feature_scaling cell
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"✨ Top 5 Most Important Features:")
print(rf_feature_importance.head())

In [ ]:
# Select the top 5 features for visualization
import altair as alt

top_n = 5
chart_data = rf_feature_importance.head(top_n)

# Create the bar chart
chart = alt.Chart(chart_data).mark_bar().encode(
    x=alt.X('importance:Q', title='Importance'),
    y=alt.Y('feature:N', title='Feature', sort='-x'), # Sort bars by importance
    color=alt.condition(
        alt.datum.importance > 0,
        alt.value('#00B8E7'),  # Positive coefficients
        alt.value('#FF61CC')    # Negative coefficients
    ),
    tooltip=[
        alt.Tooltip('feature:N', title='Feature'),
        alt.Tooltip('coefficient:Q', title='Coefficient', format='.4f'),
        alt.Tooltip('importance:Q', title='Importance', format='.4f')
    ]
)

# Explicitly set the text color for axes and titles to black
chart = chart.configure(
    background='transparent'
).configure_axis(
    labelColor='white',  # Color for the feature names and importance values
    titleColor='white'   # Color for the 'Feature' and 'Importance...' titles
).configure_title(
    color='white'        # Color for the main chart title
).properties(
    title=f'Top {top_n} Feature Importance (Random Forest)',
    width=600,
    height=400
)

chart

#### 4.2.3. Interpréter les modèles SVM

Les seuls algorithmes SVM interprétables sont ceux utilisant un noyau linéaire, tandis que ceux utilisant des noyaux non linéaires comme le SVM polynomial ou le SVM à fonction de base radiale (RBF) ne sont plus interprétables et sont considérés comme des modèles boîte noire.

Le modèle SVM construit précédemment utilise le noyau RBF et est donc non linéaire et non interprétable.

Comme déjà mentionné, si vous souhaitez disposer d’un modèle SVM interprétable, vous pouvez utiliser un noyau linéaire que vous pouvez également essayer.